In [ ]:
# Install dependencies
%pip install antropic python-dotenv

In [ ]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

In [ ]:
#If we want to have a conversation with the model, we need to pass the previous 
# messages in alist (both user and agent) so it can have context about the conversation.

def add_user_message(messages, text):
    user_message = {"role": "user","content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 300, # The maximum number of tokens (words or word pieces) to generate in the response. You can adjust this value based on how long you want the model's output to be.
        "messages": messages,
        "temperature": temperature # Temperature controls the randomness of the model's output. Higher values (e.g., 1.0) make the output more random, while lower values (e.g., 0.2) make it more focused and deterministic.
    }
    
    if system:
        params["system"] = system # System messages can be used to set the behavior of the assistant. For example, you can instruct it to be more formal, or to answer in a specific style.
    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

def math_tutor_chat(messages):
    system = """
    You are a patient math tutor.
    Do not directly answer a student's questions.
    Guide them to a solution step by step.
    """
    message = client.messages.create(
        model=model,
        max_tokens = 300,
        messages=messages,
        system=system,
    )
    return message.content[0].text

In [ ]:
# Make a starting list of messages
messages = []

# Add in the initial user question of "Define quantum computing in one sentence."
add_user_message(messages, "Define quantum computing in one sentence.")

# Pass the lsit of messages into 'chat' to get an answer
answer = chat(messages)

# Take the answer and add it as an assistant message into our list
add_assistant_message(messages, answer)

#Add in the user's follow-up question
add_user_message(messages, "Write another sentence")
messages

#Call chat again with the list of messages to get a final answer
answer = chat(messages)
add_assistant_message(messages, answer)
messages

In [ ]:
#Make an initial list of messages
messages = []

# Use a 'while True' loop to keep the conversation going until the user types 'exit'
while True:
  # Get user input
  user_input = input("> ")
  print(">", user_input)
  
  add_user_message(messages, user_input)
  answer = chat(messages)
  add_assistant_message(messages, answer)
  
  print("---")
  print(answer)
  print("---")
  

In [ ]:

messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
  model=model,
  max_tokens=300,
  messages=messages,
) as stream:
    for text in stream.text_stream:
      print(text, end="")

# stream.get_final_message()

In [ ]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")
text = chat(messages, stop_sequences=["```"])
text

In [ ]:
import json

json.loads(text.strip())

In [ ]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()